# Machine Learning Project Training

## Enzyme Function Classification from Protein Sequences

## 1. What Are We Building?

Imagine we give a computer a protein sequence:

```text
MKKIAVIGCGNIGTD...
```

The computer needs to answer:

> **What type of enzyme is this?**

We will formulate this as a **supervised multiclass classification problem**.

Our target will be the **first digit of the EC number**:

| EC Class | Enzyme family   |
| -------- | --------------- |
| **1**    | Oxidoreductases |
| **2**    | Transferases    |
| **3**    | Hydrolases      |
| **4**    | Lyases          |
| **5**    | Isomerases      |
| **6**    | Ligases         |

The EC system is hierarchical, with four levels of classification. For this project, we intentionally use only the **first level** so that the focus stays on machine-learning fundamentals. ([OUP Academic][3])

---

# 2. What This Project Covers

By the end of the project, this complete pipeline should be understood end to end:

```text
Protein Sequence
       ↓
Data Cleaning
       ↓
Exploratory Data Analysis
       ↓
Feature Engineering
       ↓
Train / Validation / Test Split
       ↓
Machine Learning Models
       ↓
Hyperparameter Tuning
       ↓
Evaluation
       ↓
Ensemble Learning
       ↓
Error Analysis
       ↓
Final Presentation
```

The goal is **not simply to obtain a high accuracy**.

The goal is to understand:

> **Why did the model make its predictions?**

---

# 3. First: What Is Supervised Learning?

In supervised learning, we have:

```text
Input X  →  Model  →  Prediction ŷ
             ↑
             |
          Training
             |
          True label y
```

For our project:

```text
X = protein sequence/features

y = enzyme class
    1, 2, 3, 4, 5, or 6
```

The model learns a relationship:

$$
f(X) \rightarrow y
$$

After training, we give it a new protein and ask it to predict its enzyme class.

---

# 4. Understanding the Dataset

Dataset:

[DanielHesslow/SwissProt-EC on Hugging Face](https://huggingface.co/datasets/DanielHesslow/SwissProt-EC)

The current dataset contains about **261K protein records**. Each record contains a protein sequence and EC annotations. ([Hugging Face][1])

A simplified record looks like:

```python
{
    "seq": "MKKIAVIGCGNIGTD...",
    "labels": [...],
    "labels_str": [
        "EC:1.-.-.-",
        "EC:1.1.-.-",
        "EC:1.1.1.-",
        "EC:1.1.1.1"
    ]
}
```

For our project we extract:

```text
EC:1.1.1.1
 ↑
first digit = 1
```

Therefore:

```python
EC:1.1.1.1 → class 1
EC:2.7.1.1 → class 2
EC:3.5.4.4 → class 3
...
```

---

# 5. Exploratory Data Analysis

Before training anything, ask:

### Dataset questions

* How many sequences are there?
* How long are the sequences?
* How many samples belong to each class?
* Are the classes balanced?
* Are there duplicate sequences?
* Are there missing sequences?
* Are there unusual amino-acid characters?

Example:

```python
df["class"].value_counts()
```

Visualize it:

```python
import matplotlib.pyplot as plt

df["class"].value_counts().sort_index().plot(kind="bar")

plt.xlabel("EC Class")
plt.ylabel("Number of Samples")
plt.title("Class Distribution")
plt.show()
```

### Key lesson

**Always understand your data before training your model.**

---

# 6. Data Cleaning

Protein sequences should contain standard amino-acid symbols.

The standard amino acids are:

```text
A C D E F G H I K L M N P Q R S T V W Y
```

Possible preprocessing:

```python
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

def is_valid_sequence(sequence):
    return all(aa in VALID_AA for aa in sequence)
```

Also investigate:

* missing values
* duplicate sequences
* extremely short sequences
* extremely long sequences
* inconsistent labels

---

# 7. Train / Validation / Test

We use:

```text
70% → Training
15% → Validation
15% → Test
```

Think of them as:

### Training

Used to **learn** the model.

### Validation

Used to **make decisions** about the model.

### Test

Used only for the **final evaluation**.

A simple rule:

> **Never use the test set to make model decisions.**

---

# 8. What Is Stratification?

Suppose we have:

```text
Class 1 → 50%
Class 2 → 20%
Class 3 → 10%
...
```

A random split could accidentally produce:

```text
Training → 70%, 15%, 15%...
Test     → 90%, 2%, 1%...
```

That would be problematic.

Stratification attempts to preserve the class proportions.

```python
train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y
)
```

---

# 9. Turning Protein Sequences into Numbers

Machine-learning models cannot directly understand:

```text
MKKIAVIGCGNIGTD...
```

We need:

```text
Protein sequence
       ↓
Numerical representation
       ↓
Machine-learning model
```

This is called **feature extraction** or **feature engineering**.

There are two broad families of representation:

```text
Hand-crafted features            Learned representations
counts and frequencies           embeddings from a pretrained model
AAC / dipeptide / tripeptide     ESM-2 / ProtBert
```

The first family is computed with rules we write ourselves. The second family is produced by a model that was already trained on millions of protein sequences.

---

# 10. Feature Set #1 - Amino Acid Composition

There are 20 standard amino acids.

For each sequence, calculate the frequency of each amino acid.

Example:

```text
Sequence:

AAAAACCCGG
```

Amino-acid frequencies:

```text
A = 5/10
C = 3/10
G = 2/10
```

The final vector has 20 values:

```text
[A%, C%, D%, E%, ..., Y%]
```

Therefore:

$$
\text{Feature dimension}=20
$$

---

# 11. Feature Set #2 - Dipeptide Composition

Instead of looking at individual amino acids, look at pairs.

For:

```text
ACDE
```

the dipeptides are:

```text
AC
CD
DE
```

There are:

$$
20^2=400
$$

possible dipeptides.

So the feature vector has up to:

```text
400 dimensions
```

---

# 12. Feature Set #3 - Tripeptides

Now consider groups of three:

```text
ACDEFG
```

gives:

```text
ACD
CDE
DEF
EFG
```

There are theoretically:

$$
20^3=8000
$$

possible tripeptides.

This provides more information but creates a much larger feature space.

---

# Feature Set #4 - Embedding Representation

Sections 10 to 12 counted things. An **embedding** compresses the same information into a small vector that is learned from the data.

```text
Sequence
   ↓
3-mer counts (8000 numbers)
   ↓
TruncatedSVD
   ↓
Embedding (e.g. 128 numbers)
   ↓
Model
```

Nothing is downloaded and no GPU is needed. scikit-learn is enough:

```python
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD

vec = CountVectorizer(analyzer="char", ngram_range=(3, 3))
X_big = vec.fit_transform(train_seqs)      # 8000 columns, sparse

svd = TruncatedSVD(n_components=128)
X_emb = svd.fit_transform(X_big)           # 128 columns = the embedding
```

Two rules:

* fit `vec` and `svd` on the **training split only**, then use `transform` for validation and test
* the embedding **replaces** the counts, it is not added to them

A pretrained protein language model such as ESM-2 gives better embeddings, but it needs `torch`, a model download and far more time. The compression above is enough for this project.

---

# 13. Why High-Dimensional Features Are a Problem

Imagine:

```text
20 features
```

versus:

```text
8,000 features
```

8,000 features can lead to:

* increased memory usage
* slower training
* irrelevant features
* overfitting
* sparse representations

Therefore we can perform **feature selection**.

---

# 14. SelectKBest

One possible approach:

```python
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif

selector = SelectKBest(
    mutual_info_classif,
    k=500
)

X_train_selected = selector.fit_transform(X_train, y_train)
```

The important concept:

> We select features that provide useful information about the target.

And importantly:

**fit the selector only on training data.**

---

# 15. Random Forest

Random Forest combines many decision trees.

Conceptually:

```text
             Dataset
                |
       ┌────────┼────────┐
       ↓        ↓        ↓
     Tree 1   Tree 2   Tree 3
       ↓        ↓        ↓
       └────────┼────────┘
                ↓
             Voting
                ↓
            Prediction
```

Each tree makes a prediction.

The forest combines those predictions.

### Advantages

* Handles nonlinear relationships
* Relatively easy to use
* Provides feature importance
* Works well with mixed feature behavior

---

# 16. LightGBM

LightGBM is a **gradient boosting decision-tree framework**.

Instead of independently creating many trees like Random Forest, boosting builds trees sequentially.

Conceptually:

```text
Tree 1
  ↓
Errors
  ↓
Tree 2
  ↓
Remaining errors
  ↓
Tree 3
  ↓
...
```

Each new tree attempts to improve the ensemble.

---

# 17. Support Vector Machine

SVM attempts to find a decision boundary that separates classes.

Simplified:

```text
Class A        |        Class B
 ● ● ●         |        ○ ○ ○
 ● ●           |        ○ ○
                ↑
           decision boundary
```

The idea is to find a boundary that separates classes with an appropriate margin.

---

# 18. Class Imbalance

Suppose:

```text
Class 1 → 50,000
Class 2 → 30,000
Class 3 → 10,000
Class 4 → 2,000
Class 5 → 1,000
Class 6 → 500
```

A model can achieve high accuracy by mostly predicting common classes.

That's why accuracy alone isn't enough.

---

# 19. Evaluation Metrics

The metrics that matter:

### Accuracy

$$
Accuracy=\frac{Correct}{Total}
$$

### Precision

Of everything predicted as a class, how much was correct?

$$
Precision=\frac{TP}{TP+FP}
$$

### Recall

Of everything actually belonging to a class, how much did we find?

$$
Recall=\frac{TP}{TP+FN}
$$

### F1

Balances precision and recall:

$$
F1=2\frac{Precision\times Recall}
{Precision+Recall}
$$

---

# 20. Macro F1

This is particularly useful for imbalanced multiclass problems.

Calculate F1 separately for every class:

```text
Class 1 → F1
Class 2 → F1
Class 3 → F1
...
Class 6 → F1
```

Then:

$$
MacroF1 =
\frac{F1_1+F1_2+\cdots+F1_6}{6}
$$

Every class receives equal importance.

---

# 21. MCC

Matthews Correlation Coefficient is another metric useful when class distributions are imbalanced.

The multiclass MCC formula does not need to be derived from scratch, but it is worth understanding:

> MCC summarizes the agreement between predicted and true labels while accounting for all parts of the confusion matrix.

---

# 22. What Is Ensemble Learning?

Instead of trusting one model:

```text
Random Forest
       +
LightGBM
       +
SVM
       ↓
Combined prediction
```

We combine several models.

The motivation:

> Different models can make different errors.

---

# 23. Hard Voting

Suppose:

```text
Random Forest → Class 2
LightGBM      → Class 2
SVM           → Class 4
```

Majority vote:

```text
Class 2
```

---

# 24. Soft Voting

Instead of only using the predicted class, use probabilities.

Example:

```text
             Class 1   Class 2   Class 3
RF             0.1       0.7       0.2
LightGBM       0.2       0.6       0.2
SVM            0.1       0.5       0.4
```

Average the probabilities.

Then select the class with the highest average probability.

---

# 25. Stacking

Stacking is a two-level model.

```text
             Original features
                    |
       ┌────────────┼────────────┐
       ↓            ↓            ↓
      RF         LightGBM       SVM
       ↓            ↓            ↓
       └────────────┼────────────┘
                    ↓
             Meta-classifier
                    ↓
               Prediction
```

Our meta-classifier:

```python
LogisticRegression(
    class_weight="balanced"
)
```

with:

```python
StackingClassifier(
    estimators=...,
    final_estimator=...
)
```

---

# 26. Avoiding Data Leakage

This is one of the most important concepts in the whole project.

Incorrect:

```text
Entire dataset
      ↓
Feature selection
      ↓
Train/test split
```

Correct:

```text
Dataset
   ↓
Train/Test split
   ↓
Training data → fit feature selector
                    ↓
                transform
                    ↓
Validation/Test → transform only
```

The test set must remain unseen.

---

# 27. Final Evaluation

The final model should produce:

### Confusion matrix

Shows:

```text
Actual class
      ↓
Predicted class
```

Example:

```text
          Predicted
          1  2  3
Actual 1  80 5  2
       2  10 60 3
       3   4  7 70
```

---

# 28. ROC Curves

For each class, treat it as:

```text
Class X vs everything else
```

This is called **one-vs-rest**.

Plot ROC curves and calculate AUC.

The idea in one sentence:

> ROC-AUC measures how well the model separates a class from the other classes across different classification thresholds.

---

# 29. Feature Importance

For Random Forest or LightGBM:

```text
Feature              Importance
--------------------------------
AA_A                 0.082
AA_L                 0.074
Dipeptide_GA         0.061
...
```

Worth discussing:

> Which sequence features appear most informative for distinguishing enzyme classes?

Don't claim that an important feature necessarily has a biological causal role.

---

# 30. Error Analysis

This is where the project becomes interesting.

Ask:

> Which classes are confused most frequently?

For example:

```text
Class 1 → Class 2
Class 3 → Class 5
```

Then investigate possible explanations.

Keep these two apart:

```text
Observed result
      ↓
Possible explanation
```

rather than claiming an unsupported biological mechanism.

---

# PROJECT TASK

# Project: Enzyme Function Classification Using Ensemble Learning

# Project Goal

Build a complete supervised machine-learning pipeline that predicts the **main enzyme class (EC 1–6)** from an amino-acid sequence.

Compare:

```text
Random Forest
      │
LightGBM
      │
SVM
      ↓
Voting Ensembles
      ↓
Stacking Ensemble
```

The objective is not only to obtain a prediction, but to understand the complete process:

> **Data → Features → Models → Evaluation → Analysis**

---

# Dataset

Use:

[SwissProt-EC - Hugging Face](https://huggingface.co/datasets/DanielHesslow/SwissProt-EC)

The dataset currently contains approximately **261K rows**. ([Hugging Face][1])

Each sequence can have hierarchical EC annotations.

For this project, extract only the **first EC digit**.

For example:

```text
EC:1.1.1.1 → 1
EC:2.7.1.1 → 2
EC:3.5.4.4 → 3
EC:4.2.1.1 → 4
EC:5.3.1.9 → 5
EC:6.1.1.1 → 6
```

The resulting problem is:

```text
Input:  Protein sequence
Output: One of 6 enzyme classes
```

---

# Required Split

Use:

```text
70% Training
15% Validation
15% Test
```

Use **stratification**.

---

# Required Feature Engineering

Implement at least **three** feature representations.

### Required minimum

**1. Amino Acid Composition**

20 features.

**2. Dipeptide Composition**

Up to 400 features.

**3. One additional feature representation**

Choose one:

* Tripeptide composition
* Pseudo-AAC
* Conjoint Triad

For tripeptides:

$$
20^3=8000
$$

features are possible, so feature selection or dimensionality reduction is needed.

### Advanced, optional

**4. Embedding representation**

Compress the tripeptide counts instead of selecting features:

```text
3-mer counts (8000)
   ↓
TruncatedSVD or others
   ↓
Embedding (e.g. 128)
```

* scikit-learn only, nothing to download
* fit on the training split only
* reuse the same three models and the same metrics

This track is not required. It is worth attempting once the required work is finished, and it usually gives the strongest single-model result.

---

# Feature Selection

For high-dimensional representations, use something such as:

```python
SelectKBest(
    mutual_info_classif,
    k=...
)
```

The exact `k` should be justified experimentally.

---

# Required Models

Train:

### Model 1

Random Forest

### Model 2

LightGBM

### Model 3

SVM

Perform hyperparameter tuning using **5-fold cross-validation on the training data**.

If the optional embedding track is used, reuse the same three models and the same evaluation. That makes the comparison fair: only the representation changes.

---

# Class Imbalance

Investigate the class distribution.

Use at least one appropriate strategy.

Possible approaches:

```text
class_weight="balanced"
```

or:

```text
SMOTE
```

or another justified method.

If using SMOTE, it must only be applied to the appropriate training folds-not to validation or test data.

---

# Required Ensembles

## 1. Hard Voting

Use:

```python
VotingClassifier(
    voting="hard"
)
```

---

## 2. Soft Voting

Use:

```python
VotingClassifier(
    voting="soft"
)
```

---

## 3. Stacking

Use:

```python
StackingClassifier(
    cv=5,
    final_estimator=LogisticRegression(
        class_weight="balanced"
    )
)
```

**Do not use an MLP/neural network as the meta-classifier.**

---

# Required Metrics

Report:

### Primary

**Macro F1**

### Additional

**MCC**

**AUPRC**

**Accuracy**

Accuracy should be treated as a reference metric, particularly if the classes are imbalanced.

---

# Required Final Analysis

The final analysis must include:

### 1. Confusion Matrix

Normalized by true class.

### 2. Per-Class Metrics

Create a table containing:

```text
Class
Precision
Recall
F1
```

### 3. ROC Curves

One-vs-rest ROC curves with AUC.

### 4. Feature Importance

Use Random Forest or LightGBM.

### 5. Error Analysis

Answer:

* Which classes are easiest to predict?
* Which are hardest?
* Which classes are commonly confused?
* Did the ensembles improve over the individual models?
* What features appear important?
* What limitations does the approach have?

---

# Optional - SHAP

Once the required work is finished:

Use SHAP to investigate the model.

For tree models, investigate:

```python
shap.TreeExplainer(...)
```

The purpose is to understand:

> Which features contribute most strongly to the model's predictions?

This is **optional**, not required.

---

# Submission Requirements

Each submission contains:

### 1. Colab Notebook or Source code repo

a notebook shall be run from beginning to end so you can present from there

### 2. Readmes or documentations

### 3. Report

Maximum:

**5 pages**

Suggested structure:

```text
1. Introduction
2. Dataset
3. Methodology
4. Feature Engineering
5. Models
6. Ensemble Methods
7. Results
9. Limitations
10. Conclusion
11. References
```

### 4. Presentation

Present your methodology, results, and interpretation on Thursday.

---

# What We Will Look For

The project is **not graded only on accuracy**.

We want to see whether the reasoning is clear:

```text
Why did you choose this feature?

Why did you choose this model?

How did you prevent data leakage?

How did you handle imbalance?

Why did one model perform differently from another?

Did the ensemble actually help?

If you used embeddings, what did they capture that the counts could not?

What mistakes did the model make?

What would you improve?
```

A model with slightly lower performance but a **well-reasoned analysis** is much more valuable than a model with a high score that nobody can explain.

# Suggested Project Workflow

Use this checklist:

```text
☐ Load dataset
☐ Understand dataset
☐ Extract EC class 1–6
☐ Clean sequences
☐ Analyze class distribution
☐ Remove/handle duplicates
☐ Stratified train/validation/test split
☐ Implement AAC features
☐ Implement dipeptide features
☐ Implement third feature set
☐ Perform feature selection where necessary
☐ Train Random Forest
☐ Train LightGBM
☐ Train SVM
☐ Perform 5-fold CV
☐ Handle class imbalance
☐ Evaluate validation performance
☐ Train hard voting
☐ Train soft voting
☐ Train stacking
☐ Evaluate on test set
☐ Generate confusion matrix
☐ Generate ROC curves
☐ Generate feature importance
☐ Perform error analysis
☐ Write report
```

Also, there are established enzyme-classification datasets specifically containing the six main EC classes; for example, EzyPred provides **9,832 enzyme sequences across EC.1–EC.6**, with redundancy controls, which could be a useful smaller alternative if the Hugging Face dataset becomes too heavy for the session. ([CSBio][4])

For a more research-oriented extension, **CARE** is another benchmark designed specifically around enzyme-function prediction and standardized evaluation, although it is substantially more involved than what belongs in a first session. ([Hugging Face][5])


[1]: https://huggingface.co/datasets/DanielHesslow/SwissProt-EC? "DanielHesslow/SwissProt-EC · Datasets at Hugging Face"
[2]: https://enzyme.expasy.org/enzyme-byclass.html? "ENZYME search by enzyme class"
[3]: https://academic.oup.com/bioinformatics/article/34/5/760/4562505? "DEEPre: sequence-based enzyme EC number prediction by deep learning | Bioinformatics | Oxford Academic"
[4]: https://www.csbio.sjtu.edu.cn/bioinf/EzyPred/Data.htm? "EzyPred: Predicting enzyme functional classes"
[5]: https://huggingface.co/datasets/arianemora/AMLD_workshop_ML4Enzymes? "arianemora/AMLD_workshop_ML4Enzymes · Datasets at Hugging Face"


## For Submission:
MY GitHub: `AlexKalll` , For private GitHub repositories, add me as a collaborator.

For Google Colab: My **Telegram:** `@Adragon_de_mello`

Submission Deadline: Wednesday, 12:00 LT